# Train RF-DETR Nano on COCO2017 (single GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roboflow/rf-detr/blob/develop/docs/cookbooks/train-coco2017.ipynb)

Downloads the full public **COCO2017** dataset and trains **RF-DETR Nano** from scratch for 40 epochs, end to
end — `pretrain_weights=None`, so no released detector checkpoint is loaded (section 4 covers what that costs
in accuracy).

This notebook uses `torch.compile` with automatic mixed precision (bf16 when supported, otherwise fp16) on any CUDA GPU. Measured on an RTX PRO 6000 (96 GB) an epoch takes
about 7 minutes, so the 40-epoch run fits in about 5 hours; on an L4 (24 GB) an epoch takes about 36 minutes and
the run needs several sessions with `resume`. A100, T4 and other CUDA GPUs run the same recipe with `batch_size`
adjusted; CPU and Apple MPS do not benefit from compilation and should use the fine-tuning cookbooks instead.
FP8 via Transformer Engine is optional and covered under the training settings in section 5: on Nano it is
within a few percent of bf16 either way, because the 14 classification heads (91 outputs) are not FP8-shaped and
stay in bf16.

Skip the RAM-disk move on a low-RAM host — it needs ~19 GB free in `/dev/shm`.

In addition to default throughput improvements, this notebook enables compilation, with host-adaptive knobs
(`num_workers`, `seed`), a batch size measured on two GPUs, and a learning-rate schedule sized for a 40-epoch
run, each explained where it is set.

Kept deliberately minimal: download, train, plot the metrics `CSVLogger` wrote during training. For dataset
preview, checkpoint saving, and inference visualization, see `fine-tune_detection.ipynb`.

## Why this notebook is simple on purpose

Recent releases moved several training-throughput fixes directly into the default configuration, so a stock run
picks them up automatically:

- **Validation forwards one model per epoch, not two.** The base-model forward pass used to run alongside the EMA
  forward every validation batch; it is now skipped when `use_ema=True` (the default).
- **`grad_accum_steps` now defaults to `1`** instead of `4`. Gradient accumulation is an explicit opt-in — raise
  `batch_size` for your GPU first, and reach for accumulation only when memory forces a smaller physical batch.
  Measured on one L4: `batch_size=16, grad_accum_steps=1` ran 27% faster per epoch than `batch_size=4,
  grad_accum_steps=4` at the same nominal effective batch, with equal mAP.
- **`eval_batch_size`** decouples the validation/test dataloaders from the training micro-batch size, so a small
  training batch no longer forces small (slower) evaluation batches.
- **COCO mAP computation reads each image's detection scores once instead of once per detection.**
- **The transformer skips materializing tensors it would otherwise reuse unchanged** on the single-feature-level
  path that Nano (and every current detection size) uses by default.
- **Pre-training sanity-check validation is skipped by default.**
- **Measured end to end:** the same recipe on rfdetr 1.9.0 takes 65 min/epoch on an RTX PRO 6000 and 97 min on
  an L4. The 1.9.0 pipeline was not GPU-bound (32 vs 22 img/s on those two cards); develop runs at 289 vs
  56 img/s.

None of this needs a flag — it is what `RFDETRNano().train()` already does. TensorBoard logging is turned off so
the notebook does not require the `loggers` extra; `CSVLogger` is always on and is what section 5 plots.

## Scope and honest expectations

COCO2017 has 118,287 training images and 5,000 validation images — two to three orders of magnitude larger than
the small Roboflow Universe datasets in the other fine-tuning cookbooks. Measured per-epoch time (training plus
validation), for RF-DETR Nano at a fixed 384 px with `seed=0`, torch 2.11.0+cu128 and Transformer Engine
2.18/2.19, on 11-12 Sep 2026 — RTX PRO 6000 Blackwell 96 GB is the mean of epochs 1-2, L4 24 GB is epoch 1:

| GPU | rfdetr 1.9.0 (bf16, batch at ceiling) | develop bf16 + compile | develop FP8 + compile |
| --- | --- | --- | --- |
| RTX PRO 6000 | 65 min @ 128 (160 OOM) | 7.2 min @ 128 (40 % mem), 7.3 min @ 288 | 7.0 min @ 128, 7.1 min @ 320 |
| L4 | 97 min @ 32 (40 is slower) | 35 min @ 32 (46 % mem), 36 min @ 64 | 40 min @ 32, 37 min @ 72 |

Epoch 0 is slower (compile warm-up, annotation index, page cache): about 8 % on the RTX PRO 6000 and up to 40 %
on an L4. Time epoch 1 onward.

> **Session limits vary; do not assume all 40 epochs will fit in one session.** Training still checkpoints every
> `checkpoint_interval` epochs (10 by default) to `last.ckpt` as a safety net: if a session merely disconnects
> and the same runtime is still alive, reopen the notebook, set `train_config.resume` to that file's path, and
> re-run the training cell to continue. On a free-tier T4/L4 the session is likely to end before 40 epochs do —
> plan on resuming. **But `OUTPUT_DIR` below is local runtime disk**: once Colab recycles the runtime (idle
> timeout, 12/24h cap, or a paid-tier disconnect), `last.ckpt` is gone with it, and there is nothing left to
> resume from. For a run that genuinely needs several sessions, point `OUTPUT_DIR` at mounted Google Drive
> (`from google.colab import drive; drive.mount("/content/drive")`, then
> `OUTPUT_DIR = "/content/drive/MyDrive/rfdetr_coco2017"`) instead of local disk, or copy `last.ckpt` off the
> runtime before it ends and copy it back before resuming.

COCO2017 also needs about 19 GB of disk once downloaded (the archives are deleted right after extraction to avoid
a ~38 GB peak). Confirm your Colab runtime has that much free space before starting the download.

## 1 - Download COCO2017

The download comes first because it is by far the longest step and needs nothing installed — only `wget` and
`unzip`, both already on a Colab runtime. Anything the environment setup below does to the session (including a
runtime restart, if pip asks for one) leaves the extracted dataset on disk untouched.

Plain `wget`/`unzip` into the standard COCO layout RF-DETR's `dataset_file="coco"` loader expects:
`coco2017/train2017/`, `coco2017/val2017/`, `coco2017/annotations/instances_{train,val}2017.json`.

Each archive is downloaded, extracted, and deleted before the next one starts, so the disk never holds more than
one archive alongside the extracted data. The `&&` chaining is deliberate: a notebook shell cell does **not** stop
on a failed command, so a bare sequence would run `unzip` on a truncated download and `rm` on a failed extraction,
then scroll past — the failure would only surface much later, as a missing-image error part-way into training.
`wget -c` resumes an interrupted download instead of restarting it, and `unzip -o` overwrites files already
extracted, so re-running this cell after a disconnected Colab session replaces any partially extracted file. The
closing `df -h` shows the disk headroom left; the extracted dataset needs about 19 GB.

In [ ]:
!mkdir -p datasets/coco2017
!cd datasets/coco2017 && wget -c -q --show-progress http://images.cocodataset.org/zips/train2017.zip && unzip -o -q train2017.zip && rm -f train2017.zip
!cd datasets/coco2017 && wget -c -q --show-progress http://images.cocodataset.org/zips/val2017.zip && unzip -o -q val2017.zip && rm -f val2017.zip
!cd datasets/coco2017 && wget -c -q --show-progress http://images.cocodataset.org/annotations/annotations_trainval2017.zip && unzip -o -q annotations_trainval2017.zip && rm -f annotations_trainval2017.zip
!df -h datasets/coco2017

Verify the extraction before spending GPU time on it. COCO2017 ships 118,287 training and 5,000 validation
images; a short count catches a truncated download or an out-of-disk extraction here, at the cost of a few
seconds, instead of several epochs later when the training loop first reaches a missing file.

In [ ]:
from pathlib import Path

COCO_ROOT = Path("datasets/coco2017")
EXPECTED_IMAGES = {"train2017": 118_287, "val2017": 5_000}

for split, expected in EXPECTED_IMAGES.items():
    found = sum(1 for _ in (COCO_ROOT / split).glob("*.jpg"))
    assert found == expected, (
        f"{COCO_ROOT / split} holds {found} images, expected {expected}. "
        "Re-run the download cell above; check the output of `df -h` for a full disk."
    )
    annotations = COCO_ROOT / "annotations" / f"instances_{split}.json"
    assert annotations.is_file(), f"Missing {annotations}. Re-run the download cell above."

print("COCO2017 is complete.")

## 2 - Set up the environment

The throughput work described above has not shipped in a tagged release yet, so this cell installs from the
`develop` branch. Once a release containing it is out, replace the git URL with a version pin.

**GPU required.** Training needs a CUDA GPU — in Colab: **Runtime → Change runtime type → GPU**. The optional
FP8 path needs one extra dependency and is covered in section 5; the install below is the automatic mixed-precision recipe.

In [ ]:
%pip install --no-build-isolation "rfdetr[train,augment,visual] @ git+https://github.com/roboflow/rf-detr.git@develop"

## 3 - Move the dataset to /dev/shm

`/dev/shm` is Linux's tmpfs RAM disk — after the move, every DataLoader read is a memory access with no storage
I/O at all. On the high-end hosts this notebook targets, RAM is plentiful (a GCP G4 shape carries 180+ GB, an
A100 Colab runtime 80+ GB; `/dev/shm` is sized to half of RAM by default), so the ~19 GB dataset fits. It is a
real win there because cloud persistent disks throttle throughput in proportion to provisioned size — a typical
100-200 GB boot disk serves 118k random-access JPEG reads per epoch slowly. JPEG *decode* cost is unaffected by
where the bytes come from; that CPU cost is what the high `num_workers` below parallelizes away.

A move (not a copy) keeps a single instance of the dataset on the machine. Two consequences to know:

- **tmpfs does not survive a runtime restart** — which is why this cell sits *after* the pip install (pip can
  restart the runtime). If the session restarts later anyway, the dataset is gone: re-run the download cell.
- **If `/dev/shm` is too small, `mv` fails part-way** and leaves files split across both locations. Recover with
  `mv /dev/shm/coco2017/* datasets/coco2017/` and train from disk — or better, avoid it: on Colab pick a
  High-RAM runtime; `df -h /dev/shm` shows capacity before you commit.

The `test -d` guard makes the cell safe to re-run: once the dataset already sits in `/dev/shm`, the `mv` is
skipped instead of nesting a second copy inside it.

In [ ]:
!df -h /dev/shm
!test -d /dev/shm/coco2017 || mv datasets/coco2017 /dev/shm/coco2017

## 4 - Load the model

`pretrain_weights=None` skips the released COCO-pretrained Nano checkpoint, so the detection transformer and its
heads start from random initialization — this notebook trains COCO2017 from scratch. The DINOv2 backbone still
loads its self-supervised hub weights: RF-DETR requests them precisely when `pretrain_weights is None`, because
a full checkpoint would otherwise carry the backbone with it.

Instantiating with `pretrain_weights=None` raises a `PretrainWeightsCompatibilityWarning` saying the model is
initialised from scratch. That is expected here; it is a warning, not an error.

> **Accuracy expectation.** From scratch in 40 epochs lands far below the published Nano checkpoint. The released
> weights come from a much longer schedule with large-scale detection pretraining behind them. Fine-tune from the
> default checkpoint (drop the `pretrain_weights` argument) whenever accuracy, not the training loop itself, is
> the goal.

`num_classes` is left at its default of `90` — the standard COCO category-id space RF-DETR's `dataset_file="coco"`
loader already uses, so it does not need to be passed explicitly.

In [ ]:
import os
from pathlib import Path

import torch

from rfdetr import RFDETRNano
from rfdetr.visualize.training import plot_loss_metrics, plot_map_metrics

if not torch.cuda.is_available():
    raise RuntimeError("This notebook requires a CUDA GPU. In Colab: Runtime -> Change runtime type -> GPU.")

print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

# Repeated from the verification cell so this section still runs on its own after a runtime restart.
COCO_ROOT = Path("datasets/coco2017")
RAMDISK_ROOT = Path("/dev/shm/coco2017")
if RAMDISK_ROOT.is_dir():
    COCO_ROOT = RAMDISK_ROOT
OUTPUT_DIR = "output/det_coco2017_nano"
EPOCHS = 40
# Capped: measured on a 48-core host, uncapped workers sat ~15% CPU-utilized while holding ~70% of RAM.
NUM_WORKERS = min(os.cpu_count() or 2, 16)
# RTX PRO 6000 / H100: 128 uses ~40 % of memory and already saturates the GPU (288 fits, same img/s).
# L4 / T4 / A10: 32. A100 40 GB: 64.
BATCH_SIZE = 128

model = RFDETRNano(compile=True, pretrain_weights=None)  # type: ignore[no-untyped-call]

## 5 - Train

`model.train()` builds the `TrainConfig` and hands the rest to PyTorch Lightning: forward pass, bipartite
matching loss, weight updates, learning-rate scheduling, and periodic COCO mAP validation.

`CSVLogger` appends one row of metrics per epoch to `output_dir/metrics.csv`, plotted below. The progress bar
reports elapsed time per epoch. Compilation adds startup overhead: time subsequent epochs separately.
First try `EPOCHS=1` to verify training, validation, EMA, and shutdown before committing to 40 epochs.

To **resume an interrupted run**, add `resume=f"{OUTPUT_DIR}/last.ckpt"` to the call below and re-run this cell.
`last.ckpt` is rewritten every epoch (the archive `checkpoint_<epoch>.ckpt` files follow `checkpoint_interval`,
10 by default), so a disconnect costs at most the epoch in flight.

### Training settings

- **`batch_size=128`** — throughput stops improving once the GPU is saturated: on the RTX PRO 6000, 128 and 288
  give the same images per second; on the L4, 32 and 64 do. Pick the batch that leaves memory headroom rather
  than the largest that fits. Two things to know when changing it: the default `lr=1e-4` was tuned around an
  effective batch of 16, and this notebook leans on cosine-plus-warmup rather than rescaling `lr` for the 8x
  larger batch — if you tune, the linear-scaling heuristic is the place to start. On a smaller GPU, reduce the
  integer batch size and adjust `grad_accum_steps` to preserve the desired effective batch; in bf16,
  `batch_size="auto"` also works and probes for the largest micro-batch that fits.
- **`num_workers`** — the default is `2`, which is fine for the few-thousand-image datasets in the other
  cookbooks and far too low for COCO2017: 118k JPEGs have to be decoded and resized every epoch, and two worker
  processes cannot keep a modern GPU fed. But more is not simply better — on a 48-core host with this exact
  configuration, 48 workers sat at ~15% CPU while eating ~70% of RAM: once the GPU is the bottleneck, extra
  workers add nothing but memory (each worker process carries its own copy of the 118k-image annotation index,
  plus `prefetch_factor` batches of decoded tensors queued per worker). Capping at 16 feeds the GPU with
  headroom; if the GPU ever waits on data, raise the cap until epoch time stops improving.
- **`seed=0`** — seeds Python, NumPy, and torch through Lightning's `seed_everything(..., workers=True)`, which
  also gives each dataloader worker a distinct, reproducible stream. Note that RF-DETR's own
  `seed_all` helper is *not* used here: besides seeding, it sets `cudnn.deterministic=True` and
  `torch.use_deterministic_algorithms(True)`, forcing slow deterministic kernels for exactly the scatter and
  grid-sample backward passes deformable attention leans on. Reproducible seeding is worth having; deterministic
  kernels are not, in a notebook about throughput.
- **`lr_scheduler="cosine"`** and **`warmup_epochs=1`** — the default schedule is `"step"` with `lr_drop=100`,
  i.e. a 10x drop after epoch 100. In a 40-epoch run that drop never happens and the learning rate stays flat
  from first step to last, so the mAP curve plateaus noisily with no final refinement. Cosine annealing is sized
  from the run's own total step count, so it decays correctly no matter how many epochs or how large a batch the
  GPU ends up with, and the one-epoch linear warmup keeps the first steps stable.

### Compilation

`RFDETRNano(compile=True)` is worth 21 % per epoch on the RTX PRO 6000 and 27 % on an L4 at the same batch,
precision and resolution. Epoch 0 pays the warm-up. Inductor's coalesce tiling analysis is disabled by the
library at the compile call (torch 2.11 asserts on it and would silently fall back to eager for the whole
forward), so no user-side flag is needed.

`multi_scale=False` trains Nano at a fixed 384 px rather than the multi-scale ladder's 544 px top scale, which
is the resolution every number above was measured at; it replaces the previous recipe's 544 px workload and may
change accuracy.

### Optional: FP8

Requires the `cuda` extra (Transformer Engine, Hopper/Ada/Blackwell, Linux x86-64), an integer `batch_size`, and
`amp_dtype="fp8"`. Measured on Nano, FP8 changes throughput by +3 % per epoch on Blackwell and −11 % on an L4,
and raises the memory ceiling to 320 from 288 (RTX PRO 6000) and to 72 from 64 (L4). Lightning warns at startup
that 14 linear layers are not FP8-shaped; those are the per-layer classification heads and they run in bf16. Use
FP8 when you are already set up for it and need the last few percent of batch headroom on Blackwell; otherwise
bf16 is the recipe.

```
%pip install --no-build-isolation "rfdetr[train,augment,visual,cuda] @ git+https://github.com/roboflow/rf-detr.git@develop"
```

The extra installs `transformer-engine[pytorch]`, which builds a PyTorch extension at install time: CUDA toolkit
headers, cuDNN, and a compiler have to be present, Blackwell needs CUDA 12.8 or later, and the kernel has to be
restarted afterwards. See the [advanced FP8 setup guide](../learn/train/advanced.md#install-the-cuda-extra).

Separately, `augmentation_backend="gpu"` moves augmentation off the CPU workers — a
rescue for CPU-starved runtimes (a 2-core Colab), and exactly backwards on this notebook's target: with the CPU
mostly idle and the GPU saturated, it would add work to the bottleneck and take it away from idle cores.

In [ ]:
model.train(
    dataset_file="coco",
    dataset_dir=COCO_ROOT,
    output_dir=OUTPUT_DIR,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    seed=0,
    lr_scheduler="cosine",
    warmup_epochs=1,
    tensorboard=False,
    progress_bar="tqdm",
    multi_scale=False,
)

## 6 - Plot CSVLogger metrics

RF-DETR's `CSVLogger` writes `val/mAP_50_95`, `val/mAP_50`, and `val/mAP_75` to `metrics.csv` (plus the
`train/loss` family and `ema_` variants), and the plotting helpers below discover those columns automatically.
`val/mAP_50_95` is the primary metric — standard COCO bounding-box mAP averaged across IoU thresholds
0.50-0.95. `val/mAP_50` rises fastest and is the clearest early signal of whether training is progressing at
all; `val/mAP_75` reflects localization precision, not just whether objects are found.

In [ ]:
from IPython.display import display
from matplotlib import pyplot as plt

METRICS_CSV = f"{OUTPUT_DIR}/metrics.csv"

loss_figure = plot_loss_metrics(METRICS_CSV)
display(loss_figure)
plt.close(loss_figure)

In [ ]:
map_figure = plot_map_metrics(METRICS_CSV)
display(map_figure)
plt.close(map_figure)